In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/oussemabouchaala23/trials/trials_sample.csv


## Installing libraries

In [11]:
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q peft accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
import transformers
print(transformers.__version__)

5.0.0


In [13]:
!pip install -q "transformers>=5.2.0" peft accelerate

In [14]:
!pip install -q transformers==5.2.0

In [15]:
import transformers
print(transformers.__version__)

5.0.0


In [16]:
!pip uninstall transformers -y
!pip install transformers==5.2.0

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)


In [19]:
!pip show transformers
!which python
!python -c "import transformers; print(transformers.__file__)"

Name: transformers
Version: 5.2.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer-slim
Required-by: kaggle-environments, peft, sentence-transformers
/usr/local/bin/python
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py


In [20]:
!pip install transformers==5.2.0 --no-deps --force-reinstall

  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0


In [21]:
!rm -rf /usr/local/lib/python3.12/dist-packages/transformers*
!pip install transformers==5.2.0

  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)


In [22]:
import transformers
print(transformers.__version__)

5.0.0


## Load the Model + LoRA Adapter

In [23]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model (the brain)
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Load LoRA adapter (the fine-tuned hat)
model = PeftModel.from_pretrained(
    base_model,
    "Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1"
)

# Load tokenizer (translates words ↔ numbers)
tokenizer = AutoTokenizer.from_pretrained(
    "Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1"
)

# Set pad token to avoid warnings
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("Model loaded on:", model.device)

ValueError: The checkpoint you are trying to load has model type `qwen3_5` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [5]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model (the brain)
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    dtype=torch.bfloat16,
    device_map="auto"
)

# Load LoRA adapter (the fine-tuned hat)
model = PeftModel.from_pretrained(
    base_model,
    "Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1"
)

# Load tokenizer (translates words ↔ numbers)
tokenizer = AutoTokenizer.from_pretrained(
    "Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1"
)

# Set pad token to avoid warnings
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("Model loaded on:", model.device)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_B.default.weight', 'base_model.model.mod

Model loaded on: cuda:0


## Test on One Sample

In [6]:
import json

clinical_text = "A Phase 3 Study of PD-L1 Expression and HbA1c Levels in Patients With NSCLC"

messages = [
    {"role": "user", "content": f"Extract all biomarker names from the following clinical text. Return ONLY a JSON.\nText: {clinical_text}"}
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)

result = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Raw output:", result)
print()
biomarkers = json.loads(result)
print("Parsed JSON:", json.dumps(biomarkers, indent=2))

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Raw output: {
  "biomarkers": [
    "PD-L1 Expression",
    "HbA1c Levels"
  ]
}


Parsed JSON: {
  "biomarkers": [
    "PD-L1 Expression",
    "HbA1c Levels"
  ]
}


## Load CSV and prepare the text

In [9]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/oussemabouchaala23/trials/trials_sample.csv")
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

Rows: 5000
Columns: ['nct_id', 'brief_title', 'official_title', 'overall_status', 'start_date', 'sponsor', 'condition', 'phase', 'intervention_name', 'intervention_type', 'primary_outcome', 'secondary_outcome', 'country', 'study_type', 'enrollment']


## Combine Text Per Trial

In [10]:
def combine_text(row):
    parts = []
    if pd.notna(row['brief_title']):
        parts.append(row['brief_title'])
    if pd.notna(row['primary_outcome']):
        parts.append(row['primary_outcome'])
    if pd.notna(row['secondary_outcome']):
        parts.append(row['secondary_outcome'])
    return " | ".join(parts) if parts else ""

df['combined_text'] = df.apply(combine_text, axis=1)

# How many have text vs empty?
has_text = (df['combined_text'] != "").sum()
print(f"Trials with text: {has_text}/{len(df)}")
print(f"Empty (no text at all): {len(df) - has_text}")
print()
# Preview one
print("Example:")
print(df['combined_text'].iloc[0][:300])

Trials with text: 5000/5000
Empty (no text at all): 0

Example:
A Study of YTS109 Cell Injection in Subjects With Relapsed/Refractory Autoimmune Hemolytic Anemia | Dose Limiting Toxicity | The incidence and frequency of treatment-emergent adverse events | Best overall response rate (BOR) of each dose group | Objective response rate (ORR) of each dose group | Tim


In [17]:
def combine_text_2(row):
    parts = []
    if pd.notna(row['brief_title']):
        parts.append(row['brief_title'])
    if pd.notna(row['official_title']):
        parts.append(row['official_title'])
    if pd.notna(row['primary_outcome']):
        parts.append(row['primary_outcome'])
    if pd.notna(row['secondary_outcome']):
        parts.append(row['secondary_outcome'])
    return " | ".join(parts) if parts else ""

df['combined_text'] = df.apply(combine_text_2, axis=1)

has_text = (df['combined_text'] != "").sum()
print(f"Trials with text: {has_text}/{len(df)}")
print(f"Empty (no text at all): {len(df) - has_text}")
print()
print("Example:")
print(df['combined_text'].iloc[0][:300])

Trials with text: 5000/5000
Empty (no text at all): 0

Example:
A Study of YTS109 Cell Injection in Subjects With Relapsed/Refractory Autoimmune Hemolytic Anemia | The Safety and Efficacy of YTS109 Cell Injection for Relapsed/Refractory Autoimmune Hemolytic Anemia Patients After Receiving Three or More Lines of Therapy. | Dose Limiting Toxicity | The incidence a


## Run Biomarker Extraction on All 5,000 Trials

In [18]:
text_lengths = df['combined_text'].str.len()
print(f"Average length: {text_lengths.mean():.0f} chars")
print(f"Max length: {text_lengths.max()} chars")
print(f"Over 3000 chars: {(text_lengths > 3000).sum()} trials")

Average length: 614 chars
Max length: 9453 chars
Over 3000 chars: 40 trials


## The Main Extraction Loop

In [20]:
import json
import time

MAX_CHARS = 5000
results = []
failed = 0

start_time = time.time()

for i, row in df.iterrows():
    text = row['combined_text'][:MAX_CHARS]
    
    # Skip empty text
    if not text.strip():
        results.append([])
        continue
    
    # Build prompt (exact format from model page)
    messages = [
        {"role": "user", "content": f"Extract all biomarker names from the following clinical text. Return ONLY a JSON.\nText: {text}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.1,
                do_sample=True
            )
        
        result_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        
        # Parse JSON
        parsed = json.loads(result_text)
        
        # Extract biomarkers list (handle different key names)
        if "biomarkers" in parsed:
            biomarkers = parsed["biomarkers"]
        elif "All_biomarkers" in parsed:
            biomarkers = parsed["All_biomarkers"]
        else:
            biomarkers = list(parsed.values())[0]
        
        # Deduplicate (case-insensitive) keeping original casing
        seen = set()
        unique = []
        for b in biomarkers:
            if b.lower() not in seen:
                seen.add(b.lower())
                unique.append(b)
        
        results.append(unique)
        
    except Exception as e:
        results.append([])
        failed += 1
    
    # Progress every 50
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start_time
        per_trial = elapsed / (i + 1)
        remaining = per_trial * (len(df) - i - 1)
        print(f"[{i+1}/{len(df)}] | Failed: {failed} | "
              f"Time: {elapsed/60:.1f}min | ETA: {remaining/60:.1f}min")
    
    # Checkpoint every 500
    if (i + 1) % 500 == 0:
        df_temp = df.iloc[:i+1].copy()
        df_temp['biomarkers'] = results
        df_temp.to_csv(f"checkpoint_{i+1}.csv", index=False)
        print(f"  >> Checkpoint saved: checkpoint_{i+1}.csv")

# Final assignment
elapsed = time.time() - start_time
df['biomarkers'] = results
print(f"\nDone! Total time: {elapsed/60:.1f}min | Failed: {failed}/{len(df)}")

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for

[50/5000] | Failed: 47 | Time: 7.0min | ETA: 689.6min


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


KeyboardInterrupt: 

In [19]:
# Test on 3 random trials to see raw output
import random
random.seed(42)
test_indices = random.sample(range(len(df)), 3)

for idx in test_indices:
    text = df.iloc[idx]['combined_text'][:5000]
    
    messages = [
        {"role": "user", "content": f"Extract all biomarker names from the following clinical text. Return ONLY a JSON.\nText: {text}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
    
    result_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    print(f"--- Trial {idx} ---")
    print(f"Input: {text[:100]}...")
    print(f"Raw output: [{result_text}]")
    print()

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


--- Trial 912 ---
Input: The Effectiveness of Flash Glucose Monitoring System on Glycemic Control in Patients With New-onset ...
Raw output: [```json
[
  "Flash Glucose Monitoring System",
  "HbA1c",
  "time in target glucose range",
  "time in hypoglycemia",
  "other biochemical indexes",
  "LDL-C",
  "TC",
  "TG"
]
```
]



Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


--- Trial 204 ---
Input: Therapeutic Potential for Aldosterone Inhibition in Duchenne Muscular Dystrophy | Therapeutic Potent...
Raw output: [```json
[
  "Aldosterone Inhibition",
  "Duchenne Muscular Dystrophy"
]
```
]

--- Trial 2253 ---
Input: AllyQuest Adherence App Intervention for HIV-positive Men Who Have Sex With Men and Transgender Wome...
Raw output: [```json
[
  "AllyQuest Adherence App Intervention",
  "AllyQuest",
  "AllyQuest Adherence App",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",
  "AllyQuest",]



In [23]:
import json
import time
import re

MAX_CHARS = 5000

def parse_model_output(raw_text):
    """Clean and parse the model's raw output into a biomarker list."""
    
    # Strip markdown backticks (```json ... ```)
    cleaned = raw_text.strip()
    cleaned = re.sub(r'^```json\s*', '', cleaned)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    cleaned = cleaned.strip()
    
    # Fix trailing comma before ] (invalid JSON)
    cleaned = re.sub(r',\s*]', ']', cleaned)
    cleaned = re.sub(r',\s*}', '}', cleaned)
    
    parsed = json.loads(cleaned)
    
    # Handle dict format {"biomarkers": [...]}
    if isinstance(parsed, dict):
        for key in parsed:
            if isinstance(parsed[key], list):
                return parsed[key]
        return []
    
    # Handle list format [...]
    if isinstance(parsed, list):
        return parsed
    
    return []

def deduplicate(biomarkers, max_items=20):
    """Remove duplicates (case-insensitive), cap at max_items."""
    seen = set()
    unique = []
    for b in biomarkers:
        if not isinstance(b, str):
            continue
        if b.lower() not in seen:
            seen.add(b.lower())
            unique.append(b)
    return unique[:max_items]

results = []
failed = 0
start_time = time.time()

for i, row in df.iterrows():
    text = row['combined_text'][:MAX_CHARS]
    
    if not text.strip():
        results.append([])
        continue
    
    messages = [
        {"role": "user", "content": f"Extract all biomarker names from the following clinical text. Return ONLY a JSON.\nText: {text}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False
            )
        
        result_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        
        biomarkers = parse_model_output(result_text)
        biomarkers = deduplicate(biomarkers)
        results.append(biomarkers)
        
    except Exception as e:
        results.append([])
        failed += 1
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start_time
        per_trial = elapsed / (i + 1)
        remaining = per_trial * (len(df) - i - 1)
        print(f"[{i+1}/{len(df)}] | Failed: {failed} | "
              f"Time: {elapsed/60:.1f}min | ETA: {remaining/60:.1f}min")
    
    if (i + 1) % 500 == 0:
        df_temp = df.iloc[:i+1].copy()
        df_temp['biomarkers'] = results
        df_temp.to_csv(f"checkpoint_{i+1}.csv", index=False)
        print(f"  >> Checkpoint saved: checkpoint_{i+1}.csv")

elapsed = time.time() - start_time
df['biomarkers'] = results
print(f"\nDone! Total time: {elapsed/60:.1f}min | Failed: {failed}/{len(df)}")

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for

[50/5000] | Failed: 3 | Time: 6.6min | ETA: 656.9min


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for

[100/5000] | Failed: 6 | Time: 13.7min | ETA: 669.6min


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for

...

[300/5000] | Failed: 23 | Time: 38.9min | ETA: 609.3min


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for

[350/5000] | Failed: 28 | Time: 45.4min | ETA: 603.6min


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for

KeyboardInterrupt: 

In [24]:
print(f"GPU memory used: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
print(f"GPU memory total: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB")

GPU memory used: 1.5 GB


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [25]:
print(f"GPU total: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

GPU total: 14.6 GB


In [26]:
import json
import time
import re

MAX_CHARS = 5000
BATCH_SIZE = 16

def parse_model_output(raw_text):
    cleaned = raw_text.strip()
    cleaned = re.sub(r'^```json\s*', '', cleaned)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    cleaned = cleaned.strip()
    cleaned = re.sub(r',\s*]', ']', cleaned)
    cleaned = re.sub(r',\s*}', '}', cleaned)
    
    parsed = json.loads(cleaned)
    
    if isinstance(parsed, dict):
        for key in parsed:
            if isinstance(parsed[key], list):
                return parsed[key]
        return []
    if isinstance(parsed, list):
        return parsed
    return []

def deduplicate(biomarkers, max_items=20):
    seen = set()
    unique = []
    for b in biomarkers:
        if not isinstance(b, str):
            continue
        if b.lower() not in seen:
            seen.add(b.lower())
            unique.append(b)
    return unique[:max_items]

def extract_batch(texts):
    """Process a batch of texts at once."""
    # Build all prompts
    all_inputs = []
    for text in texts:
        messages = [
            {"role": "user", "content": f"Extract all biomarker names from the following clinical text. Return ONLY a JSON.\nText: {text}"}
        ]
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt"
        )
        all_inputs.append(input_ids[0])
    
    # Pad to same length (left padding for generation)
    tokenizer.padding_side = "left"
    max_len = max(ids.shape[0] for ids in all_inputs)
    
    padded_ids = []
    attention_masks = []
    for ids in all_inputs:
        pad_len = max_len - ids.shape[0]
        padded = torch.cat([torch.full((pad_len,), tokenizer.pad_token_id), ids])
        mask = torch.cat([torch.zeros(pad_len, dtype=torch.long), torch.ones(ids.shape[0], dtype=torch.long)])
        padded_ids.append(padded)
        attention_masks.append(mask)
    
    input_ids = torch.stack(padded_ids).to(model.device)
    attention_mask = torch.stack(attention_masks).to(model.device)
    
    # Generate all at once
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False
        )
    
    # Decode each result
    batch_results = []
    for j in range(len(texts)):
        new_tokens = outputs[j][max_len:]
        result_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        try:
            biomarkers = parse_model_output(result_text)
            biomarkers = deduplicate(biomarkers)
            batch_results.append(biomarkers)
        except:
            batch_results.append([])
    
    return batch_results

# Main loop
results = []
failed = 0
start_time = time.time()

texts = df['combined_text'].apply(lambda x: x[:MAX_CHARS]).tolist()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    
    try:
        batch_results = extract_batch(batch)
        results.extend(batch_results)
        failed += sum(1 for r in batch_results if r == [])
    except Exception as e:
        results.extend([[] for _ in batch])
        failed += len(batch)
        print(f"  Batch {i} crashed: {e}")
    
    done = min(i + BATCH_SIZE, len(texts))
    if done % 50 < BATCH_SIZE or done == len(texts):
        elapsed = time.time() - start_time
        per_trial = elapsed / done
        remaining = per_trial * (len(texts) - done)
        print(f"[{done}/{len(texts)}] | Failed: {failed} | "
              f"Time: {elapsed/60:.1f}min | ETA: {remaining/60:.1f}min")
    
    if done % 500 < BATCH_SIZE:
        df_temp = df.iloc[:done].copy()
        df_temp['biomarkers'] = results[:done]
        df_temp.to_csv(f"checkpoint_{done}.csv", index=False)
        print(f"  >> Checkpoint saved")

elapsed = time.time() - start_time
df['biomarkers'] = results
print(f"\nDone! Total time: {elapsed/60:.1f}min | Failed: {failed}/{len(df)}")

  Batch 0 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
  Batch 16 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
  Batch 32 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
  Batch 48 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
[64/5000] | Failed: 64 | Time: 0.0min | ETA: 0.1min
  Batch 64 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
  Batch 80 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
  Batch 96 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
[112/5000] | Failed: 112 | Time: 0.0min | ETA: 0.1min
  Batch 112 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
  Batch 128 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
  Batch 144 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
[160/5000] | Failed: 160 | Time: 0.0min | ETA: 0.1min
  Batch 160 crashed: 'tokenizers.Encoding' object has no attribute 'shape'
  Batch 176 crashed: 't

In [27]:
import json
import time
import re

MAX_CHARS = 5000

def parse_model_output(raw_text):
    cleaned = raw_text.strip()
    cleaned = re.sub(r'^```json\s*', '', cleaned)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    cleaned = cleaned.strip()
    cleaned = re.sub(r',\s*]', ']', cleaned)
    cleaned = re.sub(r',\s*}', '}', cleaned)
    
    parsed = json.loads(cleaned)
    
    if isinstance(parsed, dict):
        for key in parsed:
            if isinstance(parsed[key], list):
                return parsed[key]
        return []
    if isinstance(parsed, list):
        return parsed
    return []

def deduplicate(biomarkers, max_items=20):
    seen = set()
    unique = []
    for b in biomarkers:
        if not isinstance(b, str):
            continue
        if b.lower() not in seen:
            seen.add(b.lower())
            unique.append(b)
    return unique[:max_items]

results = []
failed = 0
start_time = time.time()

for i, row in df.iterrows():
    text = row['combined_text'][:MAX_CHARS]
    
    if not text.strip():
        results.append([])
        continue
    
    messages = [
        {"role": "user", "content": f"Extract all biomarker names from the following clinical text. Return ONLY a JSON.\nText: {text}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    
    try:
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        
        result_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        
        biomarkers = parse_model_output(result_text)
        biomarkers = deduplicate(biomarkers)
        results.append(biomarkers)
        
    except Exception as e:
        results.append([])
        failed += 1
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start_time
        per_trial = elapsed / (i + 1)
        remaining = per_trial * (len(df) - i - 1)
        print(f"[{i+1}/{len(df)}] | Failed: {failed} | "
              f"Time: {elapsed/60:.1f}min | ETA: {remaining/60:.1f}min")
    
    if (i + 1) % 500 == 0:
        df_temp = df.iloc[:i+1].copy()
        df_temp['biomarkers'] = results
        df_temp.to_csv(f"checkpoint_{i+1}.csv", index=False)
        print(f"  >> Checkpoint saved")

elapsed = time.time() - start_time
df['biomarkers'] = results
print(f"\nDone! Total time: {elapsed/60:.1f}min | Failed: {failed}/{len(df)}")

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for

[50/5000] | Failed: 3 | Time: 6.6min | ETA: 653.9min


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for

KeyboardInterrupt: 

In [1]:
import json
import time
import re
import threading

MAX_CHARS = 5000

def parse_model_output(raw_text):
    cleaned = raw_text.strip()
    cleaned = re.sub(r'^```json\s*', '', cleaned)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    cleaned = cleaned.strip()
    cleaned = re.sub(r',\s*]', ']', cleaned)
    cleaned = re.sub(r',\s*}', '}', cleaned)
    parsed = json.loads(cleaned)
    if isinstance(parsed, dict):
        for key in parsed:
            if isinstance(parsed[key], list):
                return parsed[key]
        return []
    if isinstance(parsed, list):
        return parsed
    return []

def deduplicate(biomarkers, max_items=20):
    seen = set()
    unique = []
    for b in biomarkers:
        if not isinstance(b, str):
            continue
        if b.lower() not in seen:
            seen.add(b.lower())
            unique.append(b)
    return unique[:max_items]

# Load a SEPARATE model on each GPU
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model_on_gpu(gpu_id):
    base = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3.5-0.8B",
        torch_dtype=torch.bfloat16,
        device_map={"": f"cuda:{gpu_id}"}
    )
    m = PeftModel.from_pretrained(base, "Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1")
    m.eval()
    t = AutoTokenizer.from_pretrained("Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1")
    if t.pad_token is None:
        t.pad_token = t.eos_token
    return m, t

print("Loading model on GPU 0...")
model_0, tokenizer_0 = load_model_on_gpu(0)
print("Loading model on GPU 1...")
model_1, tokenizer_1 = load_model_on_gpu(1)
print("Both models ready!")

def process_chunk(indices, texts, model, tokenizer, device, result_dict, name):
    failed = 0
    for count, idx in enumerate(indices):
        text = texts[idx]
        if not text.strip():
            result_dict[idx] = []
            continue

        messages = [
            {"role": "user", "content": f"Extract all biomarker names from the following clinical text. Return ONLY a JSON.\nText: {text}"}
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt", return_dict=True
        ).to(device)

        try:
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
            result_text = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            )
            biomarkers = parse_model_output(result_text)
            biomarkers = deduplicate(biomarkers)
            result_dict[idx] = biomarkers
        except:
            result_dict[idx] = []
            failed += 1

        if (count + 1) % 50 == 0:
            print(f"  [{name}] {count+1}/{len(indices)} done | Failed: {failed}")

    print(f"  [{name}] FINISHED | Failed: {failed}/{len(indices)}")

# Split work
texts = df['combined_text'].apply(lambda x: x[:MAX_CHARS]).tolist()
mid = len(texts) // 2
indices_0 = list(range(0, mid))
indices_1 = list(range(mid, len(texts)))

result_dict = {}
start_time = time.time()

t0 = threading.Thread(target=process_chunk, args=(indices_0, texts, model_0, tokenizer_0, "cuda:0", result_dict, "GPU-0"))
t1 = threading.Thread(target=process_chunk, args=(indices_1, texts, model_1, tokenizer_1, "cuda:1", result_dict, "GPU-1"))

print("Starting both GPUs...")
t0.start()
t1.start()
t0.join()
t1.join()

elapsed = time.time() - start_time

# Reassemble in order
df['biomarkers'] = [result_dict.get(i, []) for i in range(len(texts))]
failed_total = sum(1 for v in result_dict.values() if v == [])
print(f"\nDone! Total time: {elapsed/60:.1f}min | Failed: {failed_total}/{len(texts)}")

# Save
df.to_csv("biomarkers_extracted.csv", index=False)
print("Saved: biomarkers_extracted.csv")

Loading model on GPU 0...


NameError: name 'torch' is not defined